Dirección de Rechazo en LLMs

## Referencia: "Refusal in Language Models Is Mediated by a Single Direction" (arXiv:2406.11717)

---

### Resumen Científico

Este notebook implementa el descubrimiento clave del paper: **el comportamiento de rechazo en LLMs está mediado por un subespacio unidimensional**.

#### Hallazgo Principal (§1 del paper)
Los modelos de chat (fine-tuned para seguir instrucciones y ser seguros) rechazan peticiones dañinas mediante una **única dirección** en el espacio de activaciones. Esta dirección:
- **Es necesaria**: Eliminarla previene el rechazo de instrucciones dañinas
- **Es suficiente**: Añadirla induce rechazo incluso en instrucciones inocuas

#### Implicaciones
1. Los mecanismos de seguridad actuales son sorprendentemente frágiles
2. Una simple modificación de rango-1 puede eliminar el comportamiento de rechazo
3. Abre caminos para entender y potencialmente fortalecer la alineación

---

### Estructura del Notebook

1. **Setup** - Instalación y configuración
2. **Carga del Modelo** - Parámetros dependientes del modelo
3. **Datasets** - Instrucciones dañinas vs inocuas
4. **Extracción de la Dirección de Rechazo** - §2.3 del paper
5. **Bypass de Rechazo (Ablación Direccional)** - §3.1 del paper
6. **Inducción de Rechazo (Activation Addition)** - §3.2 del paper
7. **Ortogonalización de Pesos** - §4.1 del paper

---
## 1. Setup

### Instalación de dependencias
**Nota**: Ejecutar esta celda y **reiniciar el runtime** antes de continuar.

In [ ]:
%%capture
!pip install transformers transformers_stream_generator tiktoken transformer_lens einops jaxtyping colorama
!pip install numpy==1.26.4 --force-reinstall
!pip install scikit-learn pandas --force-reinstall
!pip install transformers==4.37.2 transformers_stream_generator


In [ ]:
import torch
import functools
import einops
import requests
import pandas as pd
import io
import textwrap
import gc

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int
from colorama import Fore

print("Imports completados correctamente")

---
## 2. Carga del Modelo

### Parámetros Dependientes del Modelo

Esta sección contiene **TODOS los parámetros que debes modificar** según el modelo que uses.

| Parámetro | Descripción | Criterio de Selección |
|-----------|-------------|----------------------|
| `MODEL_PATH` | Ruta del modelo en HuggingFace | Modelo de chat/instruct fine-tuned |
| `CHAT_TEMPLATE` | Formato de conversación | Específico de cada familia de modelos |
| `LAYER` | Capa para extraer la dirección | ~40-60% de la profundidad total (ver §C.2 del paper) |
| `POS` | Posición del token | -1 (último) o posición post-instrucción |

#### Tabla de Referencia (del paper, Table 5)

| Modelo | Capas Totales | Capa Óptima (l*) | Posición (i*) |
|--------|---------------|------------------|---------------|
| Qwen-1.8B | 24 | 15 | -1 |
| Qwen-7B | 32 | 17 | -1 |
| Qwen-14B | 40 | 23 | -1 |
| Qwen-72B | 80 | 62 | -1 |
| Yi-6B | 32 | 20 | -5 |
| Yi-34B | 60 | 37 | -1 |
| Gemma-2B | 18 | 10 | -2 |
| Gemma-7B | 28 | 14 | -1 |
| Llama-2-7B | 32 | 14 | -1 |
| Llama-2-13B | 40 | 26 | -1 |
| Llama-2-70B | 80 | 21 | -1 |
| Llama-3-8B | 32 | 12 | -5 |
| Llama-3-70B | 80 | 25 | -5 |

In [ ]:
# ============================================================================
# CONFIGURACIÓN DEL MODELO - MODIFICAR SEGÚN EL MODELO A USAR
# ============================================================================

# -----------------------------------------------------------------------------
# PARÁMETRO 1: Ruta del modelo
# -----------------------------------------------------------------------------
# Criterio: Usar modelos de CHAT/INSTRUCT (no base models)
# El modelo debe haber sido fine-tuned para seguir instrucciones y rechazar
# peticiones dañinas.
#
# Ejemplos válidos:
#   - 'Qwen/Qwen-1_8B-chat'
#   - 'meta-llama/Llama-2-7b-chat-hf'
#   - 'google/gemma-7b-it'
#   - 'mistralai/Mistral-7B-Instruct-v0.1'

MODEL_PATH = 'Qwen/Qwen-1_8B-chat'  # <-- MODIFICAR AQUÍ

# -----------------------------------------------------------------------------
# PARÁMETRO 2: Dispositivo de cómputo
# -----------------------------------------------------------------------------
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando dispositivo: {DEVICE}")

# -----------------------------------------------------------------------------
# PARÁMETRO 3: Capa para extracción de la dirección de rechazo
# -----------------------------------------------------------------------------
# Criterio (§C.1 del paper):
#   - Debe estar en el rango 40-60% de las capas totales
#   - Capas muy tempranas: representaciones demasiado "crudas"
#   - Capas muy tardías (>80%): demasiado cerca del unembedding
#   - El paper usa l* < 0.8L como criterio de filtrado
#
# Para encontrar la capa óptima, se evalúa cada candidato según:
#   - bypass_score: capacidad de evitar rechazo cuando se elimina
#   - induce_score: capacidad de inducir rechazo cuando se añade
#   - kl_score: cambio mínimo en comportamiento en prompts inocuos

LAYER = 14  # <-- MODIFICAR SEGÚN MODELO (ver tabla arriba)

# -----------------------------------------------------------------------------
# PARÁMETRO 4: Posición del token
# -----------------------------------------------------------------------------
# Criterio (§2.1 del paper):
#   - Usar tokens "post-instruction" (después de la instrucción del usuario)
#   - Típicamente -1 (último token antes de la respuesta del asistente)
#   - Algunos modelos funcionan mejor con -5 (ver tabla)

POS = -1  # <-- MODIFICAR SEGÚN MODELO

# -----------------------------------------------------------------------------
# PARÁMETRO 5: Número de ejemplos de entrenamiento
# -----------------------------------------------------------------------------
# Criterio: El paper usa 128 para entrenamiento y 32 para validación
# Para esta demo simplificada usamos 32

N_INST_TRAIN = 32  # Número de instrucciones para calcular la dirección
N_INST_TEST = 32   # Número de instrucciones para evaluar

print(f"\nConfiguración:")
print(f"  Modelo: {MODEL_PATH}")
print(f"  Capa: {LAYER}")
print(f"  Posición: {POS}")
print(f"  N entrenamiento: {N_INST_TRAIN}")
print(f"  N test: {N_INST_TEST}")

In [ ]:
# ============================================================================
# CARGA DEL MODELO
# ============================================================================
# Usamos TransformerLens para tener acceso a las activaciones intermedias
# ("hooks" que permiten interceptar y modificar el forward pass)
#
# Referencia del paper (§2.1):
#   "Let x_i^(l)(t) ∈ R^{d_model} denote the residual stream activation
#    of the token at position i at the start of layer l."

print(f"Cargando modelo: {MODEL_PATH}")
print("Esto puede tardar varios minutos...")

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_PATH,
    device=DEVICE,
    dtype=torch.float16,           # FP16 para reducir memoria
    default_padding_side='left',   # Padding a la izquierda para generación
    fp16=True
)

# Configurar el tokenizer
model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = '<|extra_0|>'  # Token de padding para Qwen

# Información del modelo
print(f"\nModelo cargado exitosamente!")
print(f"  Número de capas (L): {model.cfg.n_layers}")
print(f"  Dimensión del modelo (d_model): {model.cfg.d_model}")
print(f"  Número de cabezas de atención: {model.cfg.n_heads}")
print(f"  Tamaño del vocabulario: {model.cfg.d_vocab}")

---
## 3. Chat Template

### Formato de Conversación (§2.1 del paper)

Cada modelo tiene un formato específico para estructurar las conversaciones.
El paper define **"post-instruction tokens"** como los tokens después de la instrucción
del usuario, donde el modelo formula su respuesta.

#### Templates por Familia de Modelos (Table 6 del paper)

| Modelo | Template |
|--------|----------|
| Qwen | `<\|im_start\|>user\n{instruction}<\|im_end\|>\n<\|im_start\|>assistant\n` |
| Gemma | `<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n` |
| Yi | `<\|im_start\|>user\n{instruction}<\|im_end\|>\n<\|im_start\|>assistant\n` |
| Llama-2 | `[INST] {instruction} [/INST] ` |
| Llama-3 | `<\|start_header_id\|>user<\|end_header_id\|>\n\n{instruction}<\|eot_id\|>...` |

In [ ]:
# ============================================================================
# CHAT TEMPLATES - MODIFICAR SEGÚN EL MODELO
# ============================================================================
# El template correcto es CRUCIAL para que el modelo interprete
# las instrucciones correctamente.

# -----------------------------------------------------------------------------
# Template para QWEN (usado en este ejemplo)
# -----------------------------------------------------------------------------
QWEN_CHAT_TEMPLATE = """<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

# -----------------------------------------------------------------------------
# Template para LLAMA-2
# -----------------------------------------------------------------------------
LLAMA2_CHAT_TEMPLATE = """[INST] {instruction} [/INST] """

# -----------------------------------------------------------------------------
# Template para LLAMA-3
# -----------------------------------------------------------------------------
LLAMA3_CHAT_TEMPLATE = """<|start_header_id|>user<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# -----------------------------------------------------------------------------
# Template para GEMMA
# -----------------------------------------------------------------------------
GEMMA_CHAT_TEMPLATE = """<start_of_turn>user
{instruction}<end_of_turn>
<start_of_turn>model
"""

# -----------------------------------------------------------------------------
# SELECCIONAR EL TEMPLATE ACTIVO
# -----------------------------------------------------------------------------
ACTIVE_TEMPLATE = QWEN_CHAT_TEMPLATE  # <-- MODIFICAR SEGÚN MODELO

def tokenize_instructions(
    tokenizer: AutoTokenizer,
    instructions: List[str]
) -> Int[Tensor, 'batch_size seq_len']:
    """
    Tokeniza una lista de instrucciones usando el chat template activo.

    Args:
        tokenizer: Tokenizer del modelo
        instructions: Lista de instrucciones en texto

    Returns:
        Tensor de shape (batch_size, seq_len) con los IDs de tokens
    """
    prompts = [ACTIVE_TEMPLATE.format(instruction=inst) for inst in instructions]
    return tokenizer(prompts, padding=True, truncation=False, return_tensors="pt").input_ids

# Crear función parcial con el tokenizer del modelo
tokenize_instructions_fn = functools.partial(tokenize_instructions, tokenizer=model.tokenizer)

print("Template activo configurado")

---
## 4. Datasets: Instrucciones Dañinas vs Inocuas

### Referencia: §2.2 y Apéndice A del paper

El método **difference-in-means** requiere dos conjuntos de datos contrastantes:

1. **D_harmful**: Instrucciones dañinas que el modelo debería rechazar
   - Fuentes: AdvBench, MaliciousInstruct, TDC2023, HarmBench
   
2. **D_harmless**: Instrucciones inocuas que el modelo debería responder
   - Fuente: Alpaca

El paper usa 128 ejemplos para entrenamiento y 32 para validación.
Esta demo usa 32 para entrenamiento por simplicidad.

In [ ]:
# ============================================================================
# CARGA DE DATASETS
# ============================================================================

def get_harmful_instructions():
    """
    Carga instrucciones dañinas del dataset AdvBench.

    AdvBench (Zou et al., 2023) contiene instrucciones diseñadas para
    elicitar comportamientos dañinos de LLMs, como:
    - Creación de malware
    - Instrucciones para actividades ilegales
    - Contenido discriminatorio

    Returns:
        train: Lista de instrucciones para entrenamiento
        test: Lista de instrucciones para evaluación
    """
    url = 'https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv'
    response = requests.get(url)
    dataset = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
    instructions = dataset['goal'].tolist()

    # Split 80/20 con seed fijo para reproducibilidad
    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test

def get_harmless_instructions():
    """
    Carga instrucciones inocuas del dataset Alpaca.

    Alpaca (Taori et al., 2023) contiene instrucciones generales como:
    - Preguntas de conocimiento general
    - Tareas de escritura creativa
    - Problemas matemáticos

    Returns:
        train: Lista de instrucciones para entrenamiento
        test: Lista de instrucciones para evaluación
    """
    hf_path = 'tatsu-lab/alpaca'
    dataset = load_dataset(hf_path)

    # Filtrar instrucciones que no requieren input adicional
    instructions = []
    for i in range(len(dataset['train'])):
        if dataset['train'][i]['input'].strip() == '':
            instructions.append(dataset['train'][i]['instruction'])

    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test

# Cargar datasets
print("Cargando datasets...")
harmful_inst_train, harmful_inst_test = get_harmful_instructions()
harmless_inst_train, harmless_inst_test = get_harmless_instructions()

print(f"\nDatasets cargados:")
print(f"  Harmful train: {len(harmful_inst_train)} ejemplos")
print(f"  Harmful test: {len(harmful_inst_test)} ejemplos")
print(f"  Harmless train: {len(harmless_inst_train)} ejemplos")
print(f"  Harmless test: {len(harmless_inst_test)} ejemplos")

In [ ]:
# ============================================================================
# EJEMPLOS DE INSTRUCCIONES
# ============================================================================
# Visualizar ejemplos para entender la diferencia entre ambos tipos

print("=" * 70)
print("EJEMPLOS DE INSTRUCCIONES DAÑINAS (harmful):")
print("=" * 70)
for i in range(4):
    print(f"  {i+1}. {repr(harmful_inst_train[i])}")

print("\n" + "=" * 70)
print("EJEMPLOS DE INSTRUCCIONES INOCUAS (harmless):")
print("=" * 70)
for i in range(4):
    print(f"  {i+1}. {repr(harmless_inst_train[i])}")

---
## 5. Utilidades de Generación

Funciones para generar texto con y sin intervenciones (hooks).

In [ ]:
# ============================================================================
# FUNCIONES DE GENERACIÓN
# ============================================================================

def _generate_with_hooks(
    model: HookedTransformer,
    toks: Int[Tensor, 'batch_size seq_len'],
    max_tokens_generated: int = 64,
    fwd_hooks = [],
) -> List[str]:
    """
    Genera texto token por token, aplicando hooks opcionales.

    Los hooks permiten interceptar y modificar las activaciones
    durante el forward pass. Esto es clave para:
    - Ablación direccional (eliminar la dirección de rechazo)
    - Activation addition (añadir la dirección de rechazo)

    Args:
        model: Modelo de TransformerLens
        toks: Tokens de entrada
        max_tokens_generated: Número máximo de tokens a generar
        fwd_hooks: Lista de (nombre_hook, función_hook)

    Returns:
        Lista de strings generados
    """
    all_toks = torch.zeros(
        (toks.shape[0], toks.shape[1] + max_tokens_generated),
        dtype=torch.long,
        device=toks.device
    )
    all_toks[:, :toks.shape[1]] = toks

    for i in range(max_tokens_generated):
        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_toks[:, :-max_tokens_generated + i])
            # Greedy sampling (temperatura = 0)
            next_tokens = logits[:, -1, :].argmax(dim=-1)
            all_toks[:, -max_tokens_generated + i] = next_tokens

    return model.tokenizer.batch_decode(
        all_toks[:, toks.shape[1]:],
        skip_special_tokens=True
    )

def get_generations(
    model: HookedTransformer,
    instructions: List[str],
    tokenize_instructions_fn: Callable,
    fwd_hooks = [],
    max_tokens_generated: int = 64,
    batch_size: int = 4,
) -> List[str]:
    """
    Genera respuestas para una lista de instrucciones.

    Procesa en batches para manejar memoria eficientemente.
    """
    generations = []

    for i in tqdm(range(0, len(instructions), batch_size)):
        toks = tokenize_instructions_fn(instructions=instructions[i:i+batch_size])
        generation = _generate_with_hooks(
            model,
            toks.to(DEVICE),
            max_tokens_generated=max_tokens_generated,
            fwd_hooks=fwd_hooks,
        )
        generations.extend(generation)

    return generations

print("Funciones de generación definidas")

---
## 6. Extracción de la Dirección de Rechazo

### Referencia: §2.3 del paper - "Extracting a refusal direction"

#### Método: Difference-in-Means

La técnica **difference-in-means** calcula la diferencia entre las activaciones medias
del modelo cuando procesa instrucciones dañinas vs inocuas:

$$\mathbf{r}_i^{(l)} = \boldsymbol{\mu}_i^{(l)} - \boldsymbol{\nu}_i^{(l)}$$

Donde:
- $\boldsymbol{\mu}_i^{(l)}$ = media de activaciones en capa $l$, posición $i$ para instrucciones **dañinas**
- $\boldsymbol{\nu}_i^{(l)}$ = media de activaciones en capa $l$, posición $i$ para instrucciones **inocuas**

#### Intuición
Esta dirección captura "lo que es diferente" cuando el modelo procesa contenido dañino.
Es la dirección en el espacio de activaciones que distingue harmful de harmless.

#### Normalización
Normalizamos a vector unitario: $\hat{\mathbf{r}} = \mathbf{r} / ||\mathbf{r}||$

In [ ]:
# ============================================================================
# EXTRACCIÓN DE LA DIRECCIÓN DE RECHAZO
# ============================================================================
# Implementación del método difference-in-means (§2.3 del paper)

print("Paso 1: Tokenizando instrucciones...")
harmful_toks = tokenize_instructions_fn(instructions=harmful_inst_train[:N_INST_TRAIN])
harmless_toks = tokenize_instructions_fn(instructions=harmless_inst_train[:N_INST_TRAIN])

print(f"  Tokens harmful: {harmful_toks.shape}")
print(f"  Tokens harmless: {harmless_toks.shape}")

# ============================================================================
# Ejecutar el modelo y cachear activaciones del residual stream
# ============================================================================
# El residual stream es el "flujo principal" de información en el transformer.
# Cada capa lee del residual stream, procesa, y escribe de vuelta.
#
# Cacheamos 'resid_pre', 'resid_mid', 'resid_post' de cada capa.

print("\nPaso 2: Ejecutando forward pass con caching de activaciones...")
print("  (Esto puede tardar un momento)")

harmful_logits, harmful_cache = model.run_with_cache(
    harmful_toks.to(DEVICE),
    names_filter=lambda hook_name: 'resid' in hook_name
)

harmless_logits, harmless_cache = model.run_with_cache(
    harmless_toks.to(DEVICE),
    names_filter=lambda hook_name: 'resid' in hook_name
)

print("  Forward pass completado")

# ============================================================================
# Calcular la dirección de rechazo
# ============================================================================
# Ecuación (2) del paper:
# μ_i^(l) = (1/|D_harmful|) * Σ x_i^(l)(t)  para t en D_harmful
# ν_i^(l) = (1/|D_harmless|) * Σ x_i^(l)(t) para t en D_harmless
# r_i^(l) = μ_i^(l) - ν_i^(l)

print(f"\nPaso 3: Calculando dirección de rechazo...")
print(f"  Capa: {LAYER}")
print(f"  Posición: {POS}")

# Extraer activaciones en la capa y posición especificadas
# Shape: (batch_size, seq_len, d_model) -> (batch_size, d_model)
harmful_acts = harmful_cache['resid_pre', LAYER][:, POS, :]  # (N, d_model)
harmless_acts = harmless_cache['resid_pre', LAYER][:, POS, :]  # (N, d_model)

# Calcular medias
harmful_mean_act = harmful_acts.mean(dim=0)    # μ: (d_model,)
harmless_mean_act = harmless_acts.mean(dim=0)  # ν: (d_model,)

# Dirección de rechazo: r = μ - ν
refusal_dir = harmful_mean_act - harmless_mean_act  # (d_model,)

# Normalizar a vector unitario: r̂ = r / ||r||
refusal_dir = refusal_dir / refusal_dir.norm()

print(f"\n  ✓ Dirección de rechazo calculada")
print(f"    Shape: {refusal_dir.shape}")
print(f"    Norma: {refusal_dir.norm().item():.4f} (normalizado)")
print(f"    d_model: {model.cfg.d_model}")

In [ ]:
# ============================================================================
# LIBERAR MEMORIA
# ============================================================================
# Los caches ocupan mucha memoria GPU, los liberamos después de extraer
# la dirección de rechazo

del harmful_cache, harmless_cache, harmful_logits, harmless_logits
gc.collect()
torch.cuda.empty_cache()

print("Memoria liberada")

---
## 7. Bypass de Rechazo: Ablación Direccional

### Referencia: §2.4 y §3.1 del paper - "Directional Ablation"

#### Concepto
La **ablación direccional** elimina el componente de cada activación que está alineado
con la dirección de rechazo. Matemáticamente:

$$\mathbf{a}'_l = \mathbf{a}_l - (\mathbf{a}_l \cdot \hat{\mathbf{r}}) \hat{\mathbf{r}}$$

Donde:
- $\mathbf{a}_l$ = activación original en capa $l$
- $\hat{\mathbf{r}}$ = dirección de rechazo (normalizada)
- $(\mathbf{a}_l \cdot \hat{\mathbf{r}})$ = proyección escalar de $\mathbf{a}$ sobre $\hat{\mathbf{r}}$
- $(\mathbf{a}_l \cdot \hat{\mathbf{r}}) \hat{\mathbf{r}}$ = componente vectorial de $\mathbf{a}$ en dirección $\hat{\mathbf{r}}$

#### Intuición
Imagina que cada activación tiene una "cantidad de rechazo" codificada.
La ablación direccional "borra" esa información, haciendo imposible que el modelo
exprese rechazo.

#### Implementación
Se aplica en **todas las capas** y **todas las posiciones** del residual stream.

In [ ]:
# ============================================================================
# FUNCIÓN DE ABLACIÓN DIRECCIONAL
# ============================================================================
# Implementación de la Ecuación (4) del paper

def direction_ablation_hook(
    activation: Float[Tensor, "... d_act"],
    hook: HookPoint,
    direction: Float[Tensor, "d_act"]
):
    """
    Hook que elimina el componente de la activación en una dirección dada.

    Implementa: a' = a - (a · r̂) r̂

    Donde:
    - a: activación original (... x d_act)
    - r̂: dirección a eliminar (d_act,), normalizada
    - a': activación modificada

    Args:
        activation: Tensor de activaciones del modelo
        hook: HookPoint de TransformerLens (no usado pero requerido)
        direction: Vector de dirección a eliminar (debe estar normalizado)

    Returns:
        Activación con el componente en 'direction' eliminado
    """
    # Calcular proyección: (a · r̂) r̂
    # Usamos einops para manejar tensores de dimensiones arbitrarias
    # '... d_act, d_act single -> ... single' hace el producto punto
    proj = einops.einsum(
        activation,
        direction.view(-1, 1),
        '... d_act, d_act single -> ... single'
    ) * direction

    # Restar la proyección: a' = a - proj
    return activation - proj

print("Función de ablación direccional definida")

In [ ]:
# ============================================================================
# GENERAR RESPUESTAS CON Y SIN ABLACIÓN
# ============================================================================
# Comparamos:
# 1. Baseline: Modelo sin modificar (debería rechazar)
# 2. Intervención: Modelo con ablación direccional (debería NO rechazar)

print("Configurando hooks de ablación...")

# Crear hook parcial con la dirección de rechazo
intervention_dir = refusal_dir
hook_fn = functools.partial(direction_ablation_hook, direction=intervention_dir)

# Aplicar a TODAS las capas y TODOS los tipos de residual stream
# resid_pre: antes de atención
# resid_mid: después de atención, antes de MLP
# resid_post: después de MLP
intervention_layers = list(range(model.cfg.n_layers))

fwd_hooks = [
    (utils.get_act_name(act_name, l), hook_fn)
    for l in intervention_layers
    for act_name in ['resid_pre', 'resid_mid', 'resid_post']
]

print(f"  Total de hooks: {len(fwd_hooks)}")
print(f"  Capas intervenidas: {len(intervention_layers)}")

# ============================================================================
# Generar respuestas
# ============================================================================
print("\nGenerando respuestas para instrucciones DAÑINAS...")
print("\n--- CON INTERVENCIÓN (ablación direccional) ---")
intervention_generations = get_generations(
    model,
    harmful_inst_test[:N_INST_TEST],
    tokenize_instructions_fn,
    fwd_hooks=fwd_hooks
)

print("\n--- SIN INTERVENCIÓN (baseline) ---")
baseline_generations = get_generations(
    model,
    harmful_inst_test[:N_INST_TEST],
    tokenize_instructions_fn,
    fwd_hooks=[]
)

In [ ]:
# ============================================================================
# VISUALIZAR RESULTADOS: BYPASS DE RECHAZO
# ============================================================================

print("=" * 80)
print("RESULTADOS: BYPASS DE RECHAZO EN INSTRUCCIONES DAÑINAS")
print("=" * 80)

# Mostrar algunos ejemplos
n_examples = min(5, N_INST_TEST)  # Mostrar máximo 5 ejemplos

for i in range(n_examples):
    print(f"\n{'─' * 80}")
    print(f"INSTRUCCIÓN {i+1}: {repr(harmful_inst_test[i])}")
    print(f"{'─' * 80}")

    print(Fore.GREEN + "BASELINE (sin intervención):" + Fore.RESET)
    print(textwrap.fill(
        repr(baseline_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

    print(Fore.RED + "\nCON ABLACIÓN (intervención):" + Fore.RESET)
    print(textwrap.fill(
        repr(intervention_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

---
## 8. Inducción de Rechazo: Activation Addition

### Referencia: §2.4 y §3.2 del paper - "Activation Addition"

#### Concepto
Si la ablación direccional **elimina** el rechazo, la **activation addition** puede **inducirlo**.
Añadimos la dirección de rechazo a las activaciones de instrucciones inocuas:

$$\mathbf{x}^{(l)'} \leftarrow \mathbf{x}^{(l)} + \mathbf{r}^{(l)}$$

#### Nota Importante
- Para **inducir** rechazo: añadimos la dirección ($+\mathbf{r}$)
- Para **bypass** de rechazo via addition: restamos la dirección ($-\mathbf{r}$)

#### Diferencia con Ablación
- **Ablación**: Se aplica en TODAS las capas
- **Addition**: Se aplica solo en la capa $l^*$ de donde se extrajo la dirección

#### Resultado Esperado
El modelo debería rechazar instrucciones inocuas como "List three benefits of yoga"

In [ ]:
# ============================================================================
# FUNCIÓN DE ACTIVATION ADDITION
# ============================================================================
# Implementación de la Ecuación (3) del paper

def direction_addition_hook(
    activation: Float[Tensor, "... d_act"],
    hook: HookPoint,
    direction: Float[Tensor, "d_act"],
    scale: float = 1.0
):
    """
    Hook que añade una dirección a las activaciones.

    Implementa: x' = x + scale * r

    Args:
        activation: Tensor de activaciones del modelo
        hook: HookPoint de TransformerLens
        direction: Vector de dirección a añadir
        scale: Factor de escala (1.0 = añadir dirección completa)

    Returns:
        Activación con la dirección añadida
    """
    return activation + scale * direction

print("Función de activation addition definida")

In [ ]:
# ============================================================================
# CALCULAR DIRECCIÓN NO NORMALIZADA PARA ACTIVATION ADDITION
# ============================================================================
# Para activation addition, usamos la dirección CON su magnitud original,
# no la versión normalizada. Esto es porque la magnitud captura la
# "intensidad" de la diferencia entre harmful y harmless.

print("Recalculando dirección de rechazo para activation addition...")

# Volver a ejecutar el modelo para obtener los caches
harmful_toks = tokenize_instructions_fn(instructions=harmful_inst_train[:N_INST_TRAIN])
harmless_toks = tokenize_instructions_fn(instructions=harmless_inst_train[:N_INST_TRAIN])

harmful_logits, harmful_cache = model.run_with_cache(
    harmful_toks.to(DEVICE),
    names_filter=lambda hook_name: 'resid' in hook_name
)
harmless_logits, harmless_cache = model.run_with_cache(
    harmless_toks.to(DEVICE),
    names_filter=lambda hook_name: 'resid' in hook_name
)

# Calcular dirección SIN normalizar (para activation addition)
harmful_mean_act = harmful_cache['resid_pre', LAYER][:, POS, :].mean(dim=0)
harmless_mean_act = harmless_cache['resid_pre', LAYER][:, POS, :].mean(dim=0)

# Dirección de rechazo (sin normalizar)
refusal_dir_unnormalized = harmful_mean_act - harmless_mean_act

print(f"  Norma de la dirección: {refusal_dir_unnormalized.norm().item():.4f}")

# Limpiar memoria
del harmful_cache, harmless_cache, harmful_logits, harmless_logits
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================================
# GENERAR RESPUESTAS CON INDUCCIÓN DE RECHAZO
# ============================================================================

print("Configurando hooks de activation addition...")

# Hook para añadir la dirección de rechazo
# Se aplica SOLO en la capa de donde se extrajo la dirección
addition_hook_fn = functools.partial(
    direction_addition_hook,
    direction=refusal_dir_unnormalized,
    scale=1.0  # Factor de escala (puedes ajustar)
)

# Solo aplicar en la capa LAYER, en resid_pre
addition_hooks = [
    (utils.get_act_name('resid_pre', LAYER), addition_hook_fn)
]

print(f"  Capa de intervención: {LAYER}")

# ============================================================================
# Generar respuestas para instrucciones INOCUAS
# ============================================================================
print("\nGenerando respuestas para instrucciones INOCUAS...")
print("\n--- SIN INTERVENCIÓN (baseline) ---")
baseline_harmless_generations = get_generations(
    model,
    harmless_inst_test[:N_INST_TEST],
    tokenize_instructions_fn,
    fwd_hooks=[]
)

print("\n--- CON ACTIVATION ADDITION (inducción de rechazo) ---")
induced_refusal_generations = get_generations(
    model,
    harmless_inst_test[:N_INST_TEST],
    tokenize_instructions_fn,
    fwd_hooks=addition_hooks
)

In [ ]:
# ============================================================================
# VISUALIZAR RESULTADOS: INDUCCIÓN DE RECHAZO
# ============================================================================

print("=" * 80)
print("RESULTADOS: INDUCCIÓN DE RECHAZO EN INSTRUCCIONES INOCUAS")
print("=" * 80)

n_examples = min(5, N_INST_TEST)

for i in range(n_examples):
    print(f"\n{'─' * 80}")
    print(f"INSTRUCCIÓN {i+1}: {repr(harmless_inst_test[i])}")
    print(f"{'─' * 80}")

    print(Fore.GREEN + "BASELINE (sin intervención):" + Fore.RESET)
    print(textwrap.fill(
        repr(baseline_harmless_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

    print(Fore.MAGENTA + "\nCON ACTIVATION ADDITION (rechazo inducido):" + Fore.RESET)
    print(textwrap.fill(
        repr(induced_refusal_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

---
## 9. Ortogonalización de Pesos

### Referencia: §4.1 del paper - "Weight Orthogonalization"

#### Concepto
En lugar de usar hooks en tiempo de inferencia, podemos **modificar permanentemente los pesos**
para que el modelo nunca pueda escribir la dirección de rechazo al residual stream.

Para cada matriz $W_{out}$ que escribe al residual stream:

$$W'_{out} \leftarrow W_{out} - \hat{\mathbf{r}}\hat{\mathbf{r}}^T W_{out}$$

#### Matrices a Ortogonalizar
1. **W_E**: Matriz de embeddings
2. **W_O**: Matrices de salida de atención (una por capa)
3. **W_out**: Matrices de salida de MLP (una por capa)

#### Equivalencia
Esta modificación es **matemáticamente equivalente** a la ablación direccional
en tiempo de inferencia (demostrado en §E del paper).

#### Ventaja
- No requiere hooks durante inferencia
- Modelo modificado puede exportarse/usarse normalmente

In [ ]:
# ============================================================================
# FUNCIÓN DE ORTOGONALIZACIÓN
# ============================================================================
# Implementación de la Ecuación (5) del paper

def get_orthogonalized_matrix(
    matrix: Float[Tensor, '... d_model'],
    vec: Float[Tensor, 'd_model']
) -> Float[Tensor, '... d_model']:
    """
    Ortogonaliza las columnas de una matriz respecto a un vector.

    Implementa: W' = W - r̂ r̂ᵀ W

    Esto asegura que la salida de W' nunca tenga componente en
    la dirección de r̂.

    Args:
        matrix: Matriz a ortogonalizar
        vec: Vector de dirección (debe estar normalizado)

    Returns:
        Matriz ortogonalizada
    """
    # Calcular proyección: r̂ r̂ᵀ W
    proj = einops.einsum(
        matrix,
        vec.view(-1, 1),
        '... d_model, d_model single -> ... single'
    ) * vec

    return matrix - proj

print("Función de ortogonalización definida")

In [ ]:
# ============================================================================
# ORTOGONALIZAR PESOS DEL MODELO
# ============================================================================
# ADVERTENCIA: Esta operación modifica el modelo permanentemente
# Si quieres preservar el modelo original, haz una copia primero

print("Ortogonalizando pesos del modelo...")
print("(Esto modifica el modelo permanentemente)")

# 1. Ortogonalizar matriz de embeddings
print("\n  1. Ortogonalizando W_E (embeddings)...")
model.W_E.data = get_orthogonalized_matrix(model.W_E, refusal_dir)

# 2. Ortogonalizar matrices de cada capa
print("  2. Ortogonalizando W_O y W_out de cada capa...")
for i, block in enumerate(model.blocks):
    # W_O: Matrices de salida de atención
    block.attn.W_O.data = get_orthogonalized_matrix(block.attn.W_O, refusal_dir)
    # W_out: Matrices de salida de MLP
    block.mlp.W_out.data = get_orthogonalized_matrix(block.mlp.W_out, refusal_dir)

print(f"\n✓ Modelo ortogonalizado ({model.cfg.n_layers} capas)")

In [ ]:
# ============================================================================
# GENERAR RESPUESTAS CON MODELO ORTOGONALIZADO
# ============================================================================

print("Generando respuestas con modelo ortogonalizado...")
print("(Sin hooks - la modificación está en los pesos)")

orthogonalized_generations = get_generations(
    model,
    harmful_inst_test[:N_INST_TEST],
    tokenize_instructions_fn,
    fwd_hooks=[]  # Sin hooks!
)

In [ ]:
# ============================================================================
# VISUALIZAR COMPARACIÓN COMPLETA
# ============================================================================

print("=" * 80)
print("COMPARACIÓN: BASELINE vs ABLACIÓN vs ORTOGONALIZACIÓN")
print("=" * 80)

n_examples = min(5, N_INST_TEST)

for i in range(n_examples):
    print(f"\n{'═' * 80}")
    print(f"INSTRUCCIÓN {i+1}: {repr(harmful_inst_test[i])}")
    print(f"{'═' * 80}")

    print(Fore.GREEN + "BASELINE (modelo original):" + Fore.RESET)
    print(textwrap.fill(
        repr(baseline_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

    print(Fore.RED + "\nABLACIÓN DIRECCIONAL (hooks):" + Fore.RESET)
    print(textwrap.fill(
        repr(intervention_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

    print(Fore.CYAN + "\nMODELO ORTOGONALIZADO (pesos modificados):" + Fore.RESET)
    print(textwrap.fill(
        repr(orthogonalized_generations[i]),
        width=100,
        initial_indent='  ',
        subsequent_indent='  '
    ))

---
## 10. Resumen y Guía para Otros Modelos

### Pasos para Replicar con Otro Modelo

1. **Modificar `MODEL_PATH`**: Cambiar a la ruta del nuevo modelo (debe ser chat/instruct)

2. **Modificar `ACTIVE_TEMPLATE`**: Usar el chat template correcto para la familia de modelos

3. **Modificar `LAYER`**:
   - Empezar con ~50% del número total de capas
   - Ajustar según resultados
   - Ver Table 5 del paper para valores óptimos

4. **Modificar `POS`**:
   - Típicamente -1 (último token)
   - Algunos modelos funcionan mejor con -5

5. **Modificar token de padding**: Cada modelo puede usar diferente token de padding

### Criterios de Selección de Capa (§C.1 del paper)

La capa óptima se selecciona minimizando `bypass_score` sujeto a:
- `induce_score > 0` (debe poder inducir rechazo)
- `kl_score < 0.1` (no cambiar mucho el comportamiento en prompts inocuos)
- `l < 0.8L` (no usar capas muy cercanas al unembedding)

### Recursos

- **Paper**: https://arxiv.org/abs/2406.11717
- **Código original**: https://github.com/andyrdt/refusal_direction
- **TransformerLens**: https://github.com/TransformerLensOrg/TransformerLens

In [ ]:
# ============================================================================
# GUARDAR LA DIRECCIÓN DE RECHAZO (OPCIONAL)
# ============================================================================
# Puedes guardar la dirección para usarla más tarde sin recalcular

# torch.save(refusal_dir, 'refusal_direction_qwen1.8b.pt')
# Para cargar: refusal_dir = torch.load('refusal_direction_qwen1.8b.pt')

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETADO")
print("=" * 80)
print("\nSe han demostrado las tres técnicas principales del paper:")
print("  1. ✓ Extracción de la dirección de rechazo (difference-in-means)")
print("  2. ✓ Bypass de rechazo (ablación direccional)")
print("  3. ✓ Inducción de rechazo (activation addition)")
print("  4. ✓ Ortogonalización de pesos (modificación permanente)")